In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2003_2004"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "OSQ_C.xpt"
csv_file = DATA_DIR / "OSQ_C_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Untouched source for fidelity comparison
source_df = df.copy()


print("=" * 75)
print("NHANES 2003-2004 OSQ_C XPT -> CSV")
print("=" * 75)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED RELEASED COLUMNS
# ============================================================

EXPECTED_COLUMNS = [

    "SEQN",

    "OSQ010A",
    "OSQ010B",
    "OSQ010C",

    "OSQ020A",
    "OSQ020B",
    "OSQ020C",

    "OSD030AA",
    "OSQ040AA",
    "OSD050AA",

    "OSD030AB",
    "OSQ040AB",
    "OSD050AB",

    "OSD030BA",
    "OSQ040BA",
    "OSD050BA",

    "OSD030BB",
    "OSQ040BB",
    "OSD050BB",

    "OSD030BC",
    "OSQ040BC",

    "OSD030BD",
    "OSQ040BD",
    "OSD050BD",

    "OSD030BE",
    "OSQ040BE",

    "OSD030BF",
    "OSQ040BF",

    "OSD030CA",
    "OSQ040CA",
    "OSD050CA",

    "OSD030CB",
    "OSQ040CB",
    "OSD050CB",

    "OSD030CC",
    "OSQ040CC",
    "OSD050CC",

    "OSD030CD",
    "OSQ040CD",
    "OSD050CD",

    "OSD030CE",
    "OSQ040CE",
    "OSD050CE",

    "OSD030CF",
    "OSQ040CF",

    "OSD030CG",
    "OSQ040CG",

    "OSD030CH",
    "OSQ040CH",

    "OSD030CI",
    "OSQ040CI",

    "OSD030CJ",
    "OSQ040CJ",
    "OSD050CJ",

    "OSQ060",
    "OSQ070"
]


if df.columns.tolist() == EXPECTED_COLUMNS:

    print(
        "\nPASS: Column names/order match OSQ_C."
    )

else:

    raise ValueError(
        "OSQ_C column names/order differ from expected."
    )


# ============================================================
# 4. STRUCTURE CHECK
# ============================================================

if len(df) != 5041:

    raise ValueError(
        f"Unexpected row count: {len(df)}"
    )


if len(df.columns) != 56:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: OSQ_C structure = 5,041 rows x 56 columns."
)


# ============================================================
# 5. KNOWN PANDAS/XPORT ZERO ARTIFACT CHECK
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)

numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


true_zero_count = int(
    df[numeric_cols]
    .eq(0)
    .sum()
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "True numeric zeros:",
    true_zero_count
)

print(
    "Tiny artifact occurrences:",
    tiny_total
)


if tiny_total > 0:

    print(
        "\nVariables containing tiny values:"
    )

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    # Do not make assumptions.
    # OSQ_C attached source currently has none.
    raise ValueError(
        "STOP: Tiny values unexpectedly found in OSQ_C. "
        "Do NOT automatically replace them with zero."
    )


print(
    "PASS: No tiny zero-decoding artifact exists."
)


# ============================================================
# 6. CHECK FOR GENUINE DECIMAL VALUES
# ============================================================

#
# IMPORTANT FIDELITY RULE:
#
# Never round a legitimate decimal.
#
# Scan all numeric fields first.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = df[col].dropna()

    if values.empty:
        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if fractional_mask.any():

        fractional_columns.append(
            col
        )

        print(
            f"\nGenuine fractional values in {col}:"
        )

        print(
            values[
                fractional_mask
            ].head(20)
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal values detected. "
        "These fields will NOT be converted:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional values exist "
    "in this OSQ_C XPT."
)


# ============================================================
# 7. CLEAN SEQN
# ============================================================

seqn = df["SEQN"].dropna()


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")

print(
    "Missing:",
    int(df["SEQN"].isna().sum())
)

print(
    "Duplicates:",
    int(df["SEQN"].duplicated().sum())
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)


# ============================================================
# 8. FORMAT OSQ/OSD VALUES WITHOUT .0
# ============================================================

#
# The complete XPT has already been checked above.
#
# There are no genuine decimal values.
#
# Therefore:
#
# 1.0     -> 1
# 75.0    -> 75
# 99999.0 -> 99999
#
# Missing stays blank.
#
# We DO NOT use .round().
#

OSQ_COLUMNS = [
    col
    for col in df.columns
    if col != "SEQN"
]


for col in OSQ_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: OSQ/OSD fields formatted "
    "without unnecessary .0."
)


# ============================================================
# 9. CDC CHECK - OSQ010A
# ============================================================

EXPECTED_OSQ010A = {
    1: 90,
    2: 4946,
    9: 4
}


actual = (
    df["OSQ010A"]
    .value_counts()
    .sort_index()
    .to_dict()
)


print("\n--- OSQ010A ---")


if (
    actual == EXPECTED_OSQ010A
    and
    int(df["OSQ010A"].isna().sum()) == 1
):

    print(
        "PASS: OSQ010A matches CDC."
    )

else:

    raise ValueError(
        "OSQ010A differs from CDC."
    )


# ============================================================
# 10. CDC CHECK - OSQ010B
# ============================================================

EXPECTED_OSQ010B = {
    1: 445,
    2: 4592,
    7: 1,
    9: 2
}


actual = (
    df["OSQ010B"]
    .value_counts()
    .sort_index()
    .to_dict()
)


print("\n--- OSQ010B ---")


if (
    actual == EXPECTED_OSQ010B
    and
    int(df["OSQ010B"].isna().sum()) == 1
):

    print(
        "PASS: OSQ010B matches CDC."
    )

else:

    raise ValueError(
        "OSQ010B differs from CDC."
    )


# ============================================================
# 11. CDC CHECK - OSQ010C
# ============================================================

EXPECTED_OSQ010C = {
    1: 103,
    2: 4931,
    7: 1,
    9: 5
}


actual = (
    df["OSQ010C"]
    .value_counts()
    .sort_index()
    .to_dict()
)


print("\n--- OSQ010C ---")


if (
    actual == EXPECTED_OSQ010C
    and
    int(df["OSQ010C"].isna().sum()) == 1
):

    print(
        "PASS: OSQ010C matches CDC."
    )

else:

    raise ValueError(
        "OSQ010C differs from CDC."
    )


# ============================================================
# 12. CDC CHECK - NUMBER OF HIP FRACTURES
# ============================================================

s = df["OSQ020A"]


print("\n--- OSQ020A ---")

print(
    "1-2:",
    int(s.between(1, 2).sum())
)

print(
    "9999:",
    int(s.eq(9999).sum())
)

print(
    "Missing:",
    int(s.isna().sum())
)


if (
    int(s.between(1, 2).sum()) == 88
    and
    int(s.eq(7777).sum()) == 0
    and
    int(s.eq(9999).sum()) == 2
    and
    int(s.isna().sum()) == 4951
):

    print(
        "PASS: OSQ020A matches CDC."
    )

else:

    raise ValueError(
        "OSQ020A differs from CDC."
    )


# ============================================================
# 13. CDC CHECK - NUMBER OF WRIST FRACTURES
# ============================================================

s = df["OSQ020B"]


print("\n--- OSQ020B ---")

print(
    "1-6:",
    int(s.between(1, 6).sum())
)

print(
    "Missing:",
    int(s.isna().sum())
)


if (
    int(s.between(1, 6).sum()) == 445
    and
    int(s.eq(7777).sum()) == 0
    and
    int(s.eq(9999).sum()) == 0
    and
    int(s.isna().sum()) == 4596
):

    print(
        "PASS: OSQ020B matches CDC."
    )

else:

    raise ValueError(
        "OSQ020B differs from CDC."
    )


# ============================================================
# 14. CDC CHECK - NUMBER OF SPINE FRACTURES
# ============================================================

s = df["OSQ020C"]


print("\n--- OSQ020C ---")

print(
    "1-10:",
    int(s.between(1, 10).sum())
)

print(
    "9999:",
    int(s.eq(9999).sum())
)

print(
    "Missing:",
    int(s.isna().sum())
)


if (
    int(s.between(1, 10).sum()) == 101
    and
    int(s.eq(7777).sum()) == 0
    and
    int(s.eq(9999).sum()) == 2
    and
    int(s.isna().sum()) == 4938
):

    print(
        "PASS: OSQ020C matches CDC."
    )

else:

    raise ValueError(
        "OSQ020C differs from CDC."
    )


# ============================================================
# 15. CDC CHECK - FIRST HIP FRACTURE AGE
# ============================================================

s = df["OSD030AA"]


print("\n--- OSD030AA ---")

print(
    "1-84:",
    int(s.between(1, 84).sum())
)

print(
    "85:",
    int(s.eq(85).sum())
)

print(
    "Missing:",
    int(s.isna().sum())
)


if (
    int(s.between(1, 84).sum()) == 82
    and
    int(s.eq(85).sum()) == 6
    and
    int(s.eq(77777).sum()) == 0
    and
    int(s.eq(99999).sum()) == 0
    and
    int(s.isna().sum()) == 4953
):

    print(
        "PASS: OSD030AA matches CDC."
    )

else:

    raise ValueError(
        "OSD030AA differs from CDC."
    )


# ============================================================
# 16. CDC CHECK - OSTEOPOROSIS DIAGNOSIS
# ============================================================

EXPECTED_OSQ060 = {
    1: 375,
    2: 4652,
    7: 1,
    9: 12
}


actual = (
    df["OSQ060"]
    .value_counts()
    .sort_index()
    .to_dict()
)


print("\n--- OSQ060 ---")


if (
    actual == EXPECTED_OSQ060
    and
    int(df["OSQ060"].isna().sum()) == 1
):

    print(
        "PASS: OSQ060 matches CDC."
    )

else:

    raise ValueError(
        "OSQ060 differs from CDC."
    )


# ============================================================
# 17. CDC CHECK - OSTEOPOROSIS TREATMENT
# ============================================================

EXPECTED_OSQ070 = {
    1: 265,
    2: 106,
    9: 4
}


actual = (
    df["OSQ070"]
    .value_counts()
    .sort_index()
    .to_dict()
)


print("\n--- OSQ070 ---")


if (
    actual == EXPECTED_OSQ070
    and
    int(df["OSQ070"].isna().sum()) == 4666
):

    print(
        "PASS: OSQ070 matches CDC."
    )

else:

    raise ValueError(
        "OSQ070 differs from CDC."
    )


# ============================================================
# 18. VERIFY NO VALUES CHANGED BEFORE EXPORT
# ============================================================

#
# IMPORTANT:
#
# OSQ_C has NO zero restoration.
#
# Therefore the expected values are exactly
# the original XPT values.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


value_failures = []


for col in source_df.columns:

    expected_values = (
        source_df[col]
        .to_numpy()
    )

    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        value_failures.append(
            col
        )


if value_failures:

    raise ValueError(
        "Unexpected value changes in:\n"
        + str(value_failures)
    )


print(
    "PASS: All 56 variables preserved exactly."
)


# ============================================================
# 19. FINAL TINY VALUE CHECK
# ============================================================

remaining_tiny = int(
    df.select_dtypes(
        include=[np.number]
    )
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print("\n--- FINAL TINY VALUE CHECK ---")

print(
    "Remaining tiny values:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny-value artifact exists."
    )


print(
    "PASS: No tiny zero artifact exists."
)


# ============================================================
# 20. EXPORT CSV
# ============================================================

#
# No float_format.
# No recalculation.
# No arbitrary rounding.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")
print(csv_file)


# ============================================================
# 21. RE-READ CSV
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 22. EXACT XPT -> CSV FIDELITY
# ============================================================

print("\n--- XPT -> CSV FIDELITY ---")


csv_failures = []


for col in source_df.columns:

    expected_values = (
        source_df[col]
        .to_numpy()
    )

    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV differences found in:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 56 columns match the source XPT."
)


# ============================================================
# 23. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in source_df.columns:

    source_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )

    csv_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            source_missing
            !=
            csv_missing
        )
    )


print("\n--- MISSING VALUE CHECK ---")

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 24. RAW CSV FORMAT CHECK
# ============================================================

print("\n--- RAW CSV FORMAT CHECK ---")


with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


dot_zero_count = (
    csv_text.count(".0")
)


print(
    "Occurrences of '.0':",
    dot_zero_count
)


if dot_zero_count != 0:

    raise ValueError(
        "Unexpected .0 formatting remains."
    )


print(
    "PASS: No unnecessary .0 formatting."
)


scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "Scientific notation:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation exists."
    )


print(
    "PASS: No scientific notation."
)


print("\n" + "=" * 75)
print("OSQ_C CONVERSION COMPLETE")
print("=" * 75)

NHANES 2003-2004 OSQ_C XPT -> CSV
Rows: 5,041
Columns: 56

PASS: Column names/order match OSQ_C.
PASS: OSQ_C structure = 5,041 rows x 56 columns.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact occurrences: 0
PASS: No tiny zero-decoding artifact exists.

--- FRACTIONAL VALUE CHECK ---
PASS: No genuine fractional values exist in this OSQ_C XPT.

--- SEQN ---
Missing: 0
Duplicates: 0
Minimum: 21009
Maximum: 31125

PASS: OSQ/OSD fields formatted without unnecessary .0.

--- OSQ010A ---
PASS: OSQ010A matches CDC.

--- OSQ010B ---
PASS: OSQ010B matches CDC.

--- OSQ010C ---
PASS: OSQ010C matches CDC.

--- OSQ020A ---
1-2: 88
9999: 2
Missing: 4951
PASS: OSQ020A matches CDC.

--- OSQ020B ---
1-6: 445
Missing: 4596
PASS: OSQ020B matches CDC.

--- OSQ020C ---
1-10: 101
9999: 2
Missing: 4938
PASS: OSQ020C matches CDC.

--- OSD030AA ---
1-84: 82
85: 6
Missing: 4953
PASS: OSD030AA matches CDC.

--- OSQ060 ---
PASS: OSQ060 matches CDC.

--- OSQ070 ---
PASS: OSQ070 matches CDC.
